# Sensitivity map from Perturbation experiments  -  pre-BO robustness analysis

Turns a one-variable-at-a-time perturbation screen (each row a deliberate deviation from a
**Reference** condition - lower temperature, no water, excess reagent, etc.) into a radar/spider
plot of % change in each response relative to that reference. The point is to see, before
committing to a Bayesian Optimization campaign, which conditions the reaction is fragile to (large
swings, tight control needed) versus robust to (small swings, safe to leave loose) - and whether a
perturbation that hurts the main response also hurts or helps side products.

## Imports

`pandas`/`numpy` for the data handling, `plotly.graph_objects` for the radar chart.

In [317]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

## User Settings

- `excel_file` / `sheet_name` - the perturbation screen: one row per condition, one column per
  response.
- `response_columns` / `response_labels` / `response_colors` - which response column(s) to plot as
  radar traces, their legend names, and line colors (edit the commented block above to plot a
  single response instead of two).
- `reference_condition` - the row name every other row's % sensitivity is measured against.
- `ring_levels` / `ring_colors` - the concentric background rings (the radar's % scale) and their
  fill colors, largest (-100%) first.
- `save_html` / `save_png` - whether to also write the figure to disk, and under what filenames.

In [323]:
# ======================================
# USER INPUTS
# ======================================

excel_file = "sensitivity_screen.xlsx"

sheet_name = 1

# response_columns = ["Yield"]
# response_labels = ["Yield"]
# response_colors = ["black"]
response_columns = ["Yield F", "Yield byproducts"]
response_labels = ["Yield F", "Yield byproducts"]
response_colors = ["black", "firebrick"]
reference_condition = "Reference"

title = "Reaction Sensitivity Screening"

line_color = "black"          # fallback polygon outline colour
fill_opacity = 0.0             # 0 = transparent fill inside polygon

save_html   = True
save_png    = True
output_html = "sensitivity_radar.html"
output_png  = "sensitivity_radar.png"

# Concentric reference rings: (value_%, fill_colour) — largest first
ring_levels = [-100, -75, -50, -25, 0, 25, 50, 75, 100]

ring_colors = [
    "rgba(140,0,0,0.90)",      # -100
    "rgba(190,40,40,0.85)",    # -75
    "rgba(225,100,60,0.80)",   # -50
    "rgba(245,190,120,0.75)",  # -25
    "rgba(240,240,240,0.80)",  # 0
    "rgba(190,230,180,0.75)",  # +25
    "rgba(120,200,110,0.80)",  # +50
    "rgba(50,170,70,0.85)",    # +75
    "rgba(0,120,40,0.90)"      # +100
]

# ring_colors = [
#     "rgba(120,70,170,0.60)",    # -100 %  ← medium purple
#     "rgba(140,100,190,0.60)",   # -75 %   ← soft purple
#     "rgba(100,130,210,0.60)",   # -50 %   ← periwinkle blue
#     "rgba(160,190,225,0.60)",   # -25 %   ← pale blue
#     "rgba(200,225,175,0.75)",   #   0 %   ← soft sage green
#     "rgba(160,190,225,0.60)",   # +25 %   ← pale blue
#     "rgba(100,130,210,0.60)",   # +50 %   ← periwinkle blue
#     "rgba(140,100,190,0.60)",   # +75 %   ← soft purple
#     "rgba(120,70,170,0)",    # +100 %  ← medium purple
# ]
# ======================================


## Load Data

Reads the raw perturbation screen from `excel_file`/`sheet_name` into `df` - one row per condition
(including the `Reference` row), one column per measured response.

In [324]:
df = pd.read_excel(
    excel_file,
    sheet_name=sheet_name
)

display(df)

,Condition,Yield F,Yield byproducts
0,Reference,42,23
1,high cat load,34,19
2,low cat load,26,15
3,low T,37,18
4,low conc,33,18
5,no H2O,0,23
6,2eq H2O,1,19
7,excess H2O,30,26
8,DIPEA,0,0


### Handle duplicate columns (optional)

Only fires if `df` has separate `Yield_1`/`Yield_2` replicate columns: averages them into `Yield`
and records the spread as `Yield_std`. No-op otherwise - safe to leave in regardless of your sheet's
layout.

In [11]:
if "Yield_1" in df.columns and "Yield_2" in df.columns:
    df["Yield"] = df[["Yield_1", "Yield_2"]].mean(axis=1)
    df["Yield_std"] = df[["Yield_1", "Yield_2"]].std(axis=1)

## Calculate Sensitivity (% change from reference)

For each column in `response_columns`, compares every row to the `reference_condition` row and adds
three columns: `_Delta` (raw difference), `_PercentSensitivity` (% change - what gets plotted on
the radar), and `_AbsSensitivity` (magnitude of the % change, sign-blind - used for ranking how
disruptive a perturbation is regardless of direction).

In [325]:
# Compute sensitivity for each requested response column
reference_values = {}
for col in response_columns:
    reference_values[col] = df.loc[
        df["Condition"] == reference_condition,
        col
    ].iloc[0]

    df[f"{col}_Delta"] = df[col] - reference_values[col]
    df[f"{col}_PercentSensitivity"] = (
        df[f"{col}_Delta"] / reference_values[col]
    ) * 100
    df[f"{col}_AbsSensitivity"] = df[f"{col}_Delta"].abs()

# Display the result, including all computed sensitivity columns
display(df)

,Condition,Yield F,Yield byproducts,Yield F_Delta,Yield F_PercentSensitivity,Yield F_AbsSensitivity,Yield byproducts_Delta,Yield byproducts_PercentSensitivity,Yield byproducts_AbsSensitivity
0,Reference,42,23,0,0.000000,0,0,0.000000,0
1,high cat load,34,19,-8,-19.047619,8,-4,-17.391304,4
2,low cat load,26,15,-16,-38.095238,16,-8,-34.782609,8
3,low T,37,18,-5,-11.904762,5,-5,-21.739130,5
4,low conc,33,18,-9,-21.428571,9,-5,-21.739130,5
5,no H2O,0,23,-42,-100.000000,42,0,0.000000,0
6,2eq H2O,1,19,-41,-97.619048,41,-4,-17.391304,4
7,excess H2O,30,26,-12,-28.571429,12,3,13.043478,3
8,DIPEA,0,0,-42,-100.000000,42,-23,-100.000000,23


## Remove Reference Row

Drops the `Reference` row, keeping the rest as `plot_df` - it's the 0% baseline by definition, so
it isn't an informative perturbation to draw on the radar itself.

In [326]:
plot_df = df[
    df["Condition"] != reference_condition
].copy()

display(plot_df)

,Condition,Yield F,Yield byproducts,Yield F_Delta,Yield F_PercentSensitivity,Yield F_AbsSensitivity,Yield byproducts_Delta,Yield byproducts_PercentSensitivity,Yield byproducts_AbsSensitivity
1,high cat load,34,19,-8,-19.047619,8,-4,-17.391304,4
2,low cat load,26,15,-16,-38.095238,16,-8,-34.782609,8
3,low T,37,18,-5,-11.904762,5,-5,-21.739130,5
4,low conc,33,18,-9,-21.428571,9,-5,-21.739130,5
5,no H2O,0,23,-42,-100.000000,42,0,0.000000,0
6,2eq H2O,1,19,-41,-97.619048,41,-4,-17.391304,4
7,excess H2O,30,26,-12,-28.571429,12,3,13.043478,3
8,DIPEA,0,0,-42,-100.000000,42,-23,-100.000000,23


## Rank Variables by Sensitivity

Sorts conditions by `_AbsSensitivity` of the first response in `response_columns`, so the most
disruptive perturbations (in either direction) surface first - a quick read on where the reaction
needs tight control versus where it's forgiving.

In [327]:
ranking = plot_df.sort_values(
    f"{response_columns[0]}_AbsSensitivity",
    ascending=False
)

display(
    ranking[
        ["Condition"] + [f"{col}_AbsSensitivity" for col in response_columns]
    ]
)

,Condition,Yield F_AbsSensitivity,Yield byproducts_AbsSensitivity
8,DIPEA,42,23
5,no H2O,42,0
6,2eq H2O,41,4
2,low cat load,16,8
7,excess H2O,12,3
4,low conc,9,5
1,high cat load,8,4
3,low T,5,5


## Radar Plot Function (Plotly)

Defines the reusable chart builder, `plot_radar_plotly`. `to_r` maps a % sensitivity value to a
radius so 0% sits on a middle ring and -100% sits at the center (Plotly's polar axis can't take
negative radii directly). The function then layers: the concentric background rings as a visual %
scale, one open (unfilled) polygon per response trace, and text annotations labeling each ring's
% value - then optionally writes the figure to HTML/PNG per the `save_html`/`save_png` settings.

In [328]:

def to_r(v, baseline=100):
    """Map % value to a positive radius. 0 % → baseline, -100 % → 0."""
    return v + baseline


def plot_radar_plotly(
    labels,
    value_series,
    series_names,
    line_colors=None,
    title="Radar Plot",
    ring_levels=None,
    ring_colors=None,
    save_html=False,
    save_png=False,
    html_file="radar.html",
    png_file="radar.png",
):
    """Plotly radar chart with one or more sensitivity traces."""

    if line_colors is None:
        line_colors = ["black"] * len(value_series)

    N = len(labels)
    cats_closed = labels + [labels[0]]

    fig = go.Figure()

    # ── Background rings ──────────────────────────────────────────
    if ring_levels and ring_colors:
        for level, color in zip(reversed(ring_levels), reversed(ring_colors)):
            r_val = to_r(level)
            fig.add_trace(go.Scatterpolar(
                r         = [r_val] * (N + 1),
                theta     = cats_closed,
                fill      = "toself",
                fillcolor = color,
                line      = dict(color="rgba(255,255,255,0.4)", width=0.8),
                showlegend= False,
                hoverinfo = "skip",
            ))

    # ── Data polygons ──────────────────────────────────────────────
    for name, values, color in zip(series_names, value_series, line_colors):
        vals_closed = values + [values[0]]

        fig.add_trace(go.Scatterpolar(
            r             = [to_r(v) for v in vals_closed],
            theta         = cats_closed,
            fill          = "none",
            line          = dict(color=color, width=3),
            name          = name,
            showlegend    = False,
            hovertemplate = "<b>%{theta}</b><br>%{customdata:.1f}%<extra></extra>",
            customdata    = vals_closed,
        ))

    # ── Ring % annotations ────────────────────────────────────────
    if ring_levels:
        for level in ring_levels:
            sign = "+" if level > 0 else ""
            fig.add_annotation(
                xref="paper", yref="paper",
                x=0.515,
                y=0.500 + (to_r(level) / 150) * 0.42,
                text=f"<b>{sign}{level}%</b>",
                showarrow=False,
                font=dict(size=11, color="royalblue"),
                xanchor="left", yanchor="middle",
            )

    # ── Layout ───────────────────────────────────────────────────
    fig.update_layout(
        title=dict(text=title, font=dict(size=18)),
        polar=dict(
            bgcolor="rgba(0,0,0,0)",
            radialaxis=dict(visible=False, range=[0, 150]),
            angularaxis=dict(
                tickfont=dict(size=13, color="black"),
                linecolor="white",
                gridcolor="rgba(255,255,255,0.3)",
            ),
        ),
        paper_bgcolor="white",
        margin=dict(l=80, r=80, t=80, b=80),
        width=550, height=550,
    )

    if save_html:
        fig.write_html(html_file)
        print(f"Saved → {html_file}")
    if save_png:
        fig.write_image(png_file, scale=2)
        print(f"Saved → {png_file}")

    fig.show()
    return fig

## Prepare Data and Call Radar Plot

Builds `labels` (condition names) and `value_series` (one list of `_PercentSensitivity` values per
response column) from `plot_df`, then calls `plot_radar_plotly` with those plus the User Settings
from above to produce the final chart.

In [329]:
labels = plot_df["Condition"].tolist()

value_series = [
    plot_df[f"{col}_PercentSensitivity"].tolist()
    for col in response_columns
]

fig = plot_radar_plotly(
    labels       = labels,
    value_series = value_series,
    series_names = response_labels,
    line_colors  = response_colors,
    title        = title,
    ring_levels  = ring_levels,
    ring_colors  = ring_colors,
    save_html    = save_html,
    save_png     = save_png,
    html_file    = output_html,
    png_file     = output_png,
)

Saved → sensitivity_radar.html
Saved → sensitivity_radar.png
